# Student Performance Machine Learning Analysis

This project explores student performance using supervised and
unsupervised machine learning techniques. Models are implemented
both from scratch with NumPy and with established machine learning
frameworks for comparison.

## Project Goals

- Clean and preprocess student performance data
- Predict exam scores using linear regression
- Predict pass/fail outcomes using logistic regression
- Implement a neural network from scratch
- Discover student performance patterns using clustering
- Compare custom implementations against PyTorch and scikit-learn

---
# 0. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

np.random.seed(42)
torch.manual_seed(42)

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(f"train: {train_df.shape},  test: {test_df.shape}")

---
# 1. Data Preprocessing

### 1.1 Exploratory Data Analysis

The dataset is first explored to understand its structure, feature distributions, and overall data quality. This analysis includes summary statistics, missing values, duplicates, impossible values, and potential outliers.

Visualizations are also used to examine feature distributions and relationships between student characteristics and exam performance.

In [ ]:

print("Shape:")
print(train_df.shape)

print("\nData Types:")
print(train_df.dtypes)

print("\nSummary Statistics:")
print(train_df.describe())

print("\nDataFrame Info:")
train_df.info()

print("\nMissing Values:")
missing_report = pd.DataFrame({
    "Missing Count": train_df.isnull().sum(),
    "Missing Percentage": (train_df.isnull().mean() * 100).round(2)
})

print(missing_report[missing_report["Missing Count"] > 0])

print("\nTotal Missing Values:", missing_report["Missing Count"].sum())

rows_2_or_more = (train_df.isnull().sum(axis=1) >= 2).sum()

print("Rows with 2 or more missing values:", rows_2_or_more)

duplicate_count = train_df.duplicated().sum()

print("\nDuplicate Rows:")
print(duplicate_count)

if duplicate_count > 0:
    print("\nDuplicated records:")
    print(train_df[train_df.duplicated(keep=False)])


# student_id should be positive
invalid = train_df[train_df["student_id"] <= 0]
print("Invalid student_id:")
print("None found." if invalid.empty else invalid)

# hours_studied typically in the 0-15 hours range
invalid = train_df[(train_df["hours_studied"] < 0) |
                   (train_df["hours_studied"] > 15)]
print("\nInvalid or Outlier hours_studied:")
print("None found." if invalid.empty else invalid)

# sleep_hours average per night are in the 3-12 hours range
invalid = train_df[(train_df["sleep_hours"] < 3) |
                   (train_df["sleep_hours"] > 12)]
print("\nInvalid sleep_hours:")
print("None found." if invalid.empty else invalid)

# attendance_rate must be between 0 and 100
invalid = train_df[(train_df["attendance_rate"] < 0) |
                   (train_df["attendance_rate"] > 100)]
print("\nInvalid attendance_rate:")
print("None found." if invalid.empty else invalid)

# prev_exam_score must be between 0 and 100
invalid = train_df[(train_df["prev_exam_score"] < 0) |
                   (train_df["prev_exam_score"] > 100)]
print("\nInvalid prev_exam_score:")
print("None found." if invalid.empty else invalid)

# exam_score must be between 0 and 100
invalid = train_df[(train_df["exam_score"] < 0) |
                   (train_df["exam_score"] > 100)]
print("\nInvalid exam_score:")
print("None found." if invalid.empty else invalid)

# passed must be either 0 or 1
invalid = train_df[~train_df["passed"].isin([0, 1])]
print("\nInvalid passed values:")
print("None found." if invalid.empty else invalid)

features = [
    "hours_studied",
    "sleep_hours",
    "attendance_rate",
    "prev_exam_score",
    "lucky_number",
    "exam_score"
]

for feature in features:
    plt.figure(figsize=(6,4))
    plt.hist(train_df[feature], bins=20, edgecolor="black")
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

features = [
    "hours_studied",
    "sleep_hours",
    "attendance_rate",
    "prev_exam_score",
    "lucky_number",
    "exam_score"
]

plt.figure(figsize=(10,6))
train_df[features].boxplot()
plt.title("Box Plots of Numeric Features")
plt.xticks(rotation=45)
plt.ylabel("Value")
plt.tight_layout()
plt.show()

features = [
    "hours_studied",
    "sleep_hours",
    "attendance_rate",
    "prev_exam_score",
    "lucky_number"
]

for feature in features:
    plt.figure(figsize=(6,4))
    plt.scatter(train_df[feature], train_df["exam_score"])
    plt.title(f"{feature} vs Exam Score")
    plt.xlabel(feature)
    plt.ylabel("Exam Score")
    plt.tight_layout()
    plt.show()

### EDA Findings

The exploratory analysis identified 81 rows containing missing values, with only 8 rows containing two or more missing values. Missing data can reduce the quality of the dataset and potentially affect the accuracy and reliability of the models.

A total of 6 duplicate observations were identified when comparing all features except `student_id`. The identifier was excluded because different IDs could prevent otherwise identical observations from being detected as duplicates.

Invalid values and potential outliers were also identified in features such as `hours_studied`, `attendance_rate`, and `exam_score`. These values can distort statistical analysis and introduce patterns that do not accurately represent realistic student performance.



## 1.2 Data Cleaning

The dataset is cleaned before model training to improve data quality and ensure that the models are trained on valid observations. The cleaning process addresses duplicate observations, missing values, impossible values, and outliers.

Row counts are tracked throughout the cleaning process to show how each preprocessing step affects the dataset.

In [ ]:
# === 1.2 Cleaning ===
print(f"Before cleaning: {len(train_df)} rows")

# Duplicates

duplicate_cols = train_df.columns.drop("student_id")

print(
    "Duplicates found:",
    train_df.duplicated(subset=duplicate_cols).sum()
)

clean_df = train_df.drop_duplicates(
    subset=duplicate_cols,
    keep="first"
).copy()

print(f"After duplicate removal: {len(clean_df)} rows")

# Missing Values
clean_df = clean_df[clean_df.isnull().sum(axis=1) < 2].copy()
print(f"After missing values removal: {len(clean_df)} rows")

student_ids = clean_df["student_id"].copy()

knn_df = clean_df.drop(columns=["student_id"])

imputer = KNNImputer(n_neighbors=5)

knn_df = pd.DataFrame(
    imputer.fit_transform(knn_df),
    columns=knn_df.columns,
    index=clean_df.index
)

clean_df = knn_df.copy()
clean_df.insert(0, "student_id", student_ids)


# Outliers and Impossible Values
clean_df = clean_df[
    (clean_df["student_id"] > 0) &
    (clean_df["hours_studied"] >= 0) &
    (clean_df["sleep_hours"] >= 0) &
    (clean_df["attendance_rate"].between(0, 100)) &
    (clean_df["prev_exam_score"].between(0, 100)) &
    (clean_df["exam_score"].between(0, 100)) &
    (clean_df["passed"].isin([0, 1]))
].copy()
print(f"After data range filtering: {len(clean_df)} rows")
Q1 = clean_df["hours_studied"].quantile(0.25)
Q3 = clean_df["hours_studied"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

clean_df = clean_df[
    (clean_df["hours_studied"] >= lower) &
    (clean_df["hours_studied"] <= upper)
]
print(f"After outlier removal: {len(clean_df)} rows")


### Cleaning Decisions

Missing Values: Rows with two or more missing values were removed because they represented less than 3% of the dataset. KNN imputation was then used to estimate the remaining rows with only one missing value, allowing more data to be preserved instead of removing those observations.

Outliers and Impossible Values: Impossible values, including negative hours, scores outside the 0–100 range, negative student IDs, attendance outside the 0–100 range, and pass/fail values other than 0 or 1 were removed because they do not represent realistic data and could negatively affect model performance. The IQR method was used to identify and remove outliers because it is less affected by extreme values and works well with skewed data.

Duplicates: Six duplicate observations were found when comparing all columns except student_id. The ID was excluded because each student can have a different identifier even when the rest of their data is duplicated. These six observations were removed to prevent repeated data from having an unnecessary influence on the models.

## 1.3 Feature Selection & Normalization

Relevant features are selected for model training based on their relationship to student performance. Identifier and non-predictive columns are excluded to prevent unrelated information from influencing the models.

A correlation matrix is used to examine relationships between the numerical features and target variables. The selected features are then standardized using Z-score normalization so that features with different numerical ranges contribute on a comparable scale.

The cleaned dataset is divided into an 80% training set and a 20% validation set using a fixed random seed for reproducibility. Normalization parameters are calculated using only the training data and then applied to the validation and test sets to prevent data leakage.

In [ ]:

corr = clean_df.corr(numeric_only=True)

print(corr)

plt.figure(figsize=(8,6))
plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

features = [
    "hours_studied",
    "sleep_hours",
    "attendance_rate",
    "prev_exam_score"
]

X = clean_df[features]

y_reg = clean_df["exam_score"]
y_clf = clean_df["passed"]



X_tr, X_val, y_reg_tr, y_reg_val, y_clf_tr, y_clf_val = train_test_split(
    X,
    y_reg,
    y_clf,
    test_size=0.20,
    random_state=42
)

print("Before normalization")
print("Mean:")
print(X_tr.mean())

print("\nStandard deviation:")
print(X_tr.std())

mu = X_tr.mean()
sigma = X_tr.std(ddof=0)
sigma[sigma == 0] = 1

X_tr = (X_tr - mu) / sigma
X_val = (X_val - mu) / sigma
X_test = (test_df[features] - mu) / sigma

print("\nAfter normalization")
print("Mean:")
print(X_tr.mean())

print("\nStandard deviation:")
print(X_tr.std(ddof=0))



### Feature Selection Decisions

**Excluded Features:**

- `student_id`: Excluded because it is only an identifier for each student and does not provide meaningful information for predicting exam performance. Including it could introduce irrelevant patterns into the model.

- `lucky_number`: Excluded because it is a randomly chosen number with no meaningful relationship to academic performance or exam scores. It is therefore unlikely to improve the model's predictions.

- `exam_score`: Excluded from the input features because it is the target variable being predicted. Including it would cause data leakage by giving the model access to the value it is supposed to predict.

- `passed`: Excluded because it is directly derived from `exam_score`. Including it would also cause data leakage when predicting exam scores.

**Included Features:**

- `hours_studied`: Included because study time is a relevant factor for academic performance and showed a relationship with exam scores in the exploratory analysis.

- `sleep_hours`: Included because sleep may influence concentration, memory, and academic performance, making it a potentially useful predictor of exam scores.

- `attendance_rate`: Included because class attendance can provide information about a student's exposure to course material and showed a relationship with exam performance.

- `prev_exam_score`: Included because previous academic performance can provide useful information about a student's expected performance on future exams.

---
# 2. Linear Regression 

## 2.1 Linear Regression From Scratch with NumPy

A linear regression model is implemented from scratch using NumPy to predict student exam scores from the selected normalized features.

The implementation includes the mean squared error (MSE) loss function, gradient computation, and gradient descent for optimizing the model parameters. Training loss is tracked across iterations to evaluate convergence.

After training, the model is evaluated using both training and validation MSE. The trained model is also applied to the test dataset to generate predicted exam scores for unseen students.

In [ ]:
# === 2.1 Linear regression from scratch ===
def mse_loss(y_true, y_pred):
    """
    Calculate Mean Squared Error.

    MSE = (1 / n) * sum((y_true - y_pred)^2)
    """
    return np.mean((y_true - y_pred) ** 2)


def linreg_gradients(X, y, beta):
    """
    Return the gradient of MSE with respect to beta.

    Gradient:
    (2 / n) * X.T @ (X @ beta - y)

    X includes the intercept column.
    """
    n = X.shape[0]

    # Current model predictions
    y_pred = X @ beta

    # Prediction error
    error = y_pred - y

    # Gradient for every parameter, including the intercept
    gradient = (2 / n) * (X.T @ error)

    return gradient



def train_linreg(X, y, lr=0.1, iters=1000):
    """
    Train linear regression using gradient descent.

    Returns:
        beta: final model parameters
        loss_history: MSE recorded after every iteration
    """

    # Start every parameter at zero
    beta = np.zeros(X.shape[1])

    loss_history = []

    for iteration in range(iters):

        # Calculate the gradient
        gradient = linreg_gradients(X, y, beta)

        # Gradient descent update
        beta = beta - lr * gradient

        # Calculate predictions using updated parameters
        y_pred = X @ beta

        # Save the current loss
        loss = mse_loss(y, y_pred)
        loss_history.append(loss)

        # Optional progress output
        if iteration % 100 == 0:
            print(f"Iteration {iteration}: MSE = {loss:.4f}")

    return beta, loss_history

# Target column
target_col = "exam_score"

# Select only the features that should be used by the model
feature_cols = [
    "hours_studied",
    "attendance_rate",
    "sleep_hours",
    "prev_exam_score"
]

# Create X and y from the CLEANED DataFrame
X = clean_df[feature_cols].to_numpy(dtype=float)
y = clean_df[target_col].to_numpy(dtype=float)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

# Create shuffled row indexes
indices = np.random.permutation(len(X))

# Use 80% for training
split_index = int(0.80 * len(X))

train_indices = indices[:split_index]
val_indices = indices[split_index:]

X_train = X[train_indices]
X_val = X[val_indices]

y_train = y[train_indices]
y_val = y[val_indices]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

# Calculate normalization values using only training data
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

# Prevent division by zero for constant columns
train_std[train_std == 0] = 1

# Normalize training and validation features
X_train_normalized = (X_train - train_mean) / train_std
X_val_normalized = (X_val - train_mean) / train_std


# Add a column of ones for the intercept
X_train_design = np.column_stack(
    [np.ones(len(X_train_normalized)), X_train_normalized]
)

X_val_design = np.column_stack(
    [np.ones(len(X_val_normalized)), X_val_normalized]
)

print("Training design matrix:", X_train_design.shape)
print("Validation design matrix:", X_val_design.shape)

beta, loss_history = train_linreg(
    X_train_design,
    y_train,
    lr=0.01,
    iters=1000
)

plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Linear Regression Loss Curve")
plt.grid(True)
plt.show()

# Make predictions
linreg_train_predictions = X_train_design @ beta
linreg_val_predictions = X_val_design @ beta

# Calculate final errors
train_mse = mse_loss(y_train, linreg_train_predictions)
validation_mse = mse_loss(y_val, linreg_val_predictions)

print("\nFinal model parameters:")
print("Intercept:", beta[0])

for feature, coefficient in zip(feature_cols, beta[1:]):
    print(f"{feature}: {coefficient}")

print("\nTraining MSE:", train_mse)
print("Validation MSE:", validation_mse)

# Select the same features and in the same order
X_test = test_df[feature_cols].to_numpy(dtype=float)

# Normalize using the TRAINING statistics
X_test_normalized = (X_test - train_mean) / train_std

# Add the intercept column
X_test_design = np.column_stack(
    [np.ones(len(X_test_normalized)), X_test_normalized]
)

# Predict exam scores
test_predictions = X_test_design @ beta

print("\nFirst 5 predicted exam scores:")
print(test_predictions[:5])


### Model Performance

The NumPy linear regression model converged during training, with the
loss curve showing how the MSE decreased across iterations.

- Training MSE: 27.09
- Validation MSE: 38.18

The model was then applied to the unseen test dataset to generate
exam-score predictions.

## 2.2 PyTorch Verification 

In [ ]:
# === 2.2 PyTorch verification ===
X_train_tensor = torch.tensor(
    X_train_design,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val_design,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
).reshape(-1, 1)

# X_train_design already contains the intercept column,
# so bias=False prevents PyTorch from adding another intercept.
torch_model = nn.Linear(
    in_features=X_train_design.shape[1],
    out_features=1,
    bias=False
)

# Start PyTorch with the same zero parameters as NumPy
with torch.no_grad():
    torch_model.weight.zero_()

criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    torch_model.parameters(),
    lr=0.01
)

torch_loss_history = []

iters = 1000

for iteration in range(iters):

    # Make predictions
    predictions = torch_model(X_train_tensor)

    # Calculate MSE
    loss = criterion(predictions, y_train_tensor)

    # Clear gradients from the previous iteration
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update the parameters
    optimizer.step()

    # Save the loss
    torch_loss_history.append(loss.item())

    if iteration % 100 == 0:
        print(
            f"Iteration {iteration}: "
            f"PyTorch MSE = {loss.item():.4f}"
        )

plt.figure(figsize=(8, 5))
plt.plot(torch_loss_history)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("PyTorch Linear Regression Loss Curve")
plt.grid(True)
plt.show()

# Get Pytorch model parameters
torch_beta = (
    torch_model.weight
    .detach()
    .numpy()
    .flatten()
)

#Calculate Pytorch training and validation MSE
with torch.no_grad():

    torch_train_predictions = torch_model(X_train_tensor)
    torch_val_predictions = torch_model(X_val_tensor)

    torch_train_mse = criterion(
        torch_train_predictions,
        y_train_tensor
    ).item()

    torch_validation_mse = criterion(
        torch_val_predictions,
        y_val_tensor
    ).item()


print("\nPyTorch parameters:")
print(torch_beta)

print("\nPyTorch Training MSE:", torch_train_mse)
print("PyTorch Validation MSE:", torch_validation_mse)

# Side by Side comparison of NumPy and PyTorch results
parameter_names = ["Intercept"] + feature_cols

comparison_table = pd.DataFrame({
    "Parameter": parameter_names,
    "NumPy": beta,
    "PyTorch": torch_beta,
    "Absolute Difference": np.abs(beta - torch_beta)
})

display(comparison_table)

# Side by Side comparison of NumPy and PyTorch MSE
mse_comparison = pd.DataFrame({
    "Metric": [
        "Training MSE",
        "Validation MSE"
    ],
    "NumPy": [
        train_mse,
        validation_mse
    ],
    "PyTorch": [
        torch_train_mse,
        torch_validation_mse
    ],
    "Absolute Difference": [
        abs(train_mse - torch_train_mse),
        abs(validation_mse - torch_validation_mse)
    ]
})

display(mse_comparison)


## 2.3 Polynomial Regression 

In [ ]:
# === 2.3 Polynomial regression ===

# Choosing the first feature (hours_studied) for polynomial regression
poly_feature_index = 0

X_train_poly_base = X_train_normalized[:, poly_feature_index]
X_val_poly_base = X_val_normalized[:, poly_feature_index]


# Create polynomial feature columns, then standardize each power column
def make_polynomial_features(x, degree, mean=None, std=None, fit=False):
    raw = np.column_stack([x ** power for power in range(degree + 1)])

    if fit:
        # column 0 is the constant (all ones) -> don't scale it
        mean = raw[:, 1:].mean(axis=0)
        std = raw[:, 1:].std(axis=0)
        std[std == 0] = 1  # avoid divide-by-zero

    scaled = raw.copy()
    scaled[:, 1:] = (raw[:, 1:] - mean) / std

    return scaled, mean, std


degrees = [2, 5, 10]

polynomial_results = []
polynomial_models = {}

plt.figure(figsize=(9, 6))

# Plot the training data
plt.scatter(
    X_train_poly_base,
    y_train,
    label="Training data",
    alpha=0.6
)

for degree in degrees:

    # Fit scaling on TRAIN, then apply the same scaling to val/curve
    X_train_poly, poly_mean, poly_std = make_polynomial_features(
        X_train_poly_base, degree, fit=True
    )

    X_val_poly, _, _ = make_polynomial_features(
        X_val_poly_base, degree, mean=poly_mean, std=poly_std
    )

    # Use the linear regression function from Section 2.1
    poly_beta, poly_loss_history = train_linreg(
        X_train_poly,
        y_train,
        lr=0.01,        
        iters=1000
    )

    # Save everything needed for this polynomial model
    polynomial_models[degree] = {
    "beta": poly_beta,
    "mean": poly_mean,
    "std": poly_std
    }

    # Predictions
    train_predictions = X_train_poly @ poly_beta
    val_predictions = X_val_poly @ poly_beta

    # MSE
    degree_train_mse = mse_loss(y_train, train_predictions)
    degree_val_mse = mse_loss(y_val, val_predictions)

    polynomial_results.append({
        "Degree": degree,
        "Training MSE": degree_train_mse,
        "Validation MSE": degree_val_mse
    })

    # Create smooth x-values for the fitted curve
    x_curve = np.linspace(
        X_train_poly_base.min(),
        X_train_poly_base.max(),
        300
    )

    # Create polynomial features for the curve using TRAIN scaling (poly_mean/poly_std)
    X_curve_poly, _, _ = make_polynomial_features(
        x_curve,
        degree,
        mean=poly_mean,
        std=poly_std
    )

    y_curve = X_curve_poly @ poly_beta

    plt.plot(
        x_curve,
        y_curve,
        label=f"Degree {degree}"
    )


plt.xlabel(f"Normalized {feature_cols[poly_feature_index]}")
plt.ylabel("Exam Score")
plt.title("Polynomial Regression Fitted Curves")
plt.legend()
plt.grid(True)
plt.show()


# Display train and validation MSE for each degree
polynomial_results_df = pd.DataFrame(polynomial_results)

display(polynomial_results_df)


# Find the degree with the lowest validation MSE
best_row = polynomial_results_df.loc[
    polynomial_results_df["Validation MSE"].idxmin()
]

best_degree = int(best_row["Degree"])

print("Best degree based on validation MSE:", best_degree)

### Polynomial Regression Analysis

The degree 2 polynomial model achieved the best validation performance,
with an MSE of 65.00 compared with 67.13 for degree 5 and 68.10 for
degree 10.

Although the higher-degree models achieved slightly lower training MSE,
they performed worse on the validation data. This suggests that increasing
the model complexity did not improve its ability to generalize to unseen
data and may indicate overfitting.

Degree 2 provides the best balance between model complexity and validation
performance among the polynomial degrees tested. These results demonstrate
the bias-variance tradeoff, where increasing model complexity can improve
training performance while reducing performance on unseen data.

---
# 3. Logistic Regression

Logistic regression is used to predict whether a student will pass or fail based on the same selected features used for the linear regression models.

The model is implemented from scratch using NumPy, including the sigmoid function, its derivative, binary cross-entropy loss, gradient computation, and gradient descent. The custom implementation is later compared with a PyTorch implementation to verify its performance.

The target variable is `passed`, where `0` represents a failing outcome and `1` represents a passing outcome.

## 3.1 Logistic Regression From Scratch with NumPy

A logistic regression model is implemented from scratch using NumPy to predict whether a student will pass or fail.

The implementation includes the sigmoid activation function, binary cross-entropy loss, gradient computation, and gradient descent for optimizing the model parameters. Training loss is recorded across iterations to evaluate convergence.

After training, predicted probabilities are converted into binary predictions using a decision threshold of 0.5. Model performance is evaluated using training and validation accuracy.

In [ ]:
# === 3.1a Sigmoid + BCE (REUSED IN PART 4) ===
def sigmoid(z):
    # Clip values to prevent overflow in exp()
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def bce_loss(y_true, y_hat):
    """Binary cross-entropy for a single sample or array."""
    eps = 1e-8  # Prevent log(0)

    y_hat = np.clip(y_hat, eps, 1 - eps)

    return -np.mean(
        y_true * np.log(y_hat)
        + (1 - y_true) * np.log(1 - y_hat)
    )

In [165]:
for z in [-2, -1, 0, 1, 2, 3, 5]:
    print(f"sigma({z:2d}) = {sigmoid(z):.4f},  sigma'({z:2d}) = {sigmoid_derivative(z):.4f}")

sigma(-2) = 0.1192,  sigma'(-2) = 0.1050
sigma(-1) = 0.2689,  sigma'(-1) = 0.1966
sigma( 0) = 0.5000,  sigma'( 0) = 0.2500
sigma( 1) = 0.7311,  sigma'( 1) = 0.1966
sigma( 2) = 0.8808,  sigma'( 2) = 0.1050
sigma( 3) = 0.9526,  sigma'( 3) = 0.0452
sigma( 5) = 0.9933,  sigma'( 5) = 0.0066


In [ ]:
# === 3.1b Logistic regression from scratch ===
def train_logreg(X, y, lr=0.1, iters=1000):
    """
    Train logistic regression using gradient descent.

    Returns:
        beta: trained model parameters
        loss_history: BCE loss at every iteration
    """

    n = X.shape[0]

    # Initialize all parameters to zero
    beta = np.zeros(X.shape[1])

    loss_history = []

    for iteration in range(iters):

        # Linear model output
        z = X @ beta

        # Convert outputs into probabilities
        y_hat = sigmoid(z)

        # Logistic regression gradient
        gradient = (1 / n) * (X.T @ (y_hat - y))

        # Update parameters
        beta = beta - lr * gradient

        # Calculate loss after updating parameters
        y_hat_updated = sigmoid(X @ beta)
        loss = bce_loss(y, y_hat_updated)

        loss_history.append(loss)

    return beta, loss_history

# Use the binary target from clean_df
y_passed = clean_df["passed"].to_numpy(dtype=float)

# Use the same training and validation rows from Part 2
y_train_log = y_passed[train_indices]
y_val_log = y_passed[val_indices]

log_beta, log_loss_history = train_logreg(
    X_train_design,
    y_train_log,
    lr=0.1,
    iters=1000
)

plt.figure(figsize=(8, 5))
plt.plot(log_loss_history)
plt.xlabel("Iteration")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Logistic Regression Loss Curve")
plt.grid(True)
plt.show()

# Training probabilities
train_probabilities = sigmoid(X_train_design @ log_beta)

# Validation probabilities
val_probabilities = sigmoid(X_val_design @ log_beta)

# Convert probabilities into predictions
train_class_predictions = (
    train_probabilities >= 0.5
).astype(int)

val_class_predictions = (
    val_probabilities >= 0.5
).astype(int)

train_accuracy = np.mean(
    train_class_predictions == y_train_log
)

validation_accuracy = np.mean(
    val_class_predictions == y_val_log
)

print("Final logistic regression parameters:")
print(log_beta)

print(f"\nTraining accuracy: {train_accuracy:.4f}")
print(f"Validation accuracy: {validation_accuracy:.4f}")

print(f"Training accuracy: {train_accuracy * 100:.2f}%")
print(f"Validation accuracy: {validation_accuracy * 100:.2f}%")

## 3.2 PyTorch Verification

A second logistic regression model is implemented using PyTorch to verify the results of the custom NumPy implementation.

The PyTorch model uses binary cross-entropy loss and an optimizer to learn the model parameters. After training, its validation accuracy and learned parameters are compared with the NumPy implementation to confirm that both approaches produce similar results.

In [ ]:
# === 3.2 PyTorch verification ===
X_train_tensor = torch.tensor(
    X_train_design,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_design,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_log,
    dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val_log,
    dtype=torch.float32
).reshape(-1, 1)

# Model
torch_model = nn.Linear(
    X_train_design.shape[1],
    1,
    bias=False
)

with torch.no_grad():
    torch_model.weight.zero_()

#Loss Optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    torch_model.parameters(),
    lr=0.1
)

#Train 
torch_loss_history = []

for _ in range(1000):

    logits = torch_model(X_train_tensor)

    loss = criterion(
        logits,
        y_train_tensor
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    torch_loss_history.append(loss.item())

plt.figure(figsize=(8,5))

plt.plot(torch_loss_history)

plt.xlabel("Iteration")
plt.ylabel("Binary Cross Entropy")

plt.title("PyTorch Logistic Regression Loss")

plt.grid(True)

plt.show()

with torch.no_grad():

    train_prob = torch.sigmoid(
        torch_model(X_train_tensor)
    )

    val_prob = torch.sigmoid(
        torch_model(X_val_tensor)
    )

    train_pred = (
        train_prob >= 0.5
    ).float()

    val_pred = (
        val_prob >= 0.5
    ).float()

    torch_train_accuracy = (
        train_pred.eq(y_train_tensor)
        .float()
        .mean()
        .item()
    )

    torch_validation_accuracy = (
        val_pred.eq(y_val_tensor)
        .float()
        .mean()
        .item()
    )

torch_beta = (
    torch_model.weight
    .detach()
    .numpy()
    .flatten()
)

comparison = pd.DataFrame({
    "Parameter": ["Intercept"] + feature_cols,
    "NumPy": log_beta,
    "PyTorch": torch_beta,
    "Difference": np.abs(log_beta - torch_beta)
})

display(comparison)

accuracy_table = pd.DataFrame({
    "Model": ["NumPy", "PyTorch"],
    "Validation Accuracy": [
        validation_accuracy,
        torch_validation_accuracy
    ]
})

display(accuracy_table)

## 3.3 Decision Threshold Analysis

The effect of different classification thresholds is evaluated to understand how the decision boundary influences model predictions. Validation accuracy is compared using thresholds of 0.3, 0.5, and 0.7.

Although 0.5 is commonly used as the default threshold for binary classification, different thresholds may be preferred depending on the importance of false positives and false negatives.

In [ ]:
# === 3.3 Thresholds ===
thresholds = [0.3, 0.5, 0.7]

threshold_results = []

for threshold in thresholds:

    predictions = (val_probabilities >= threshold).astype(int)

    accuracy = np.mean(predictions == y_val_log)

    threshold_results.append({
        "Threshold": threshold,
        "Validation Accuracy": accuracy
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results)

### Threshold and Loss Function Analysis

**Decision Thresholds:** Thresholds of 0.3, 0.5, and 0.7 were evaluated. The 0.7 threshold achieved the highest validation accuracy at 91.80%, compared with 90.16% at 0.5 and 86.89% at 0.3. A lower threshold classifies more students as passing, while a higher threshold requires greater confidence before predicting a passing outcome. Depending on the application, a threshold other than 0.5 may be preferred when reducing false positives or false negatives is more important than maximizing overall accuracy.

**Cross-Entropy Loss:** Binary cross-entropy is more appropriate than MSE for this classification task because it is designed to evaluate predicted probabilities against binary outcomes. It penalizes confident incorrect predictions more strongly and provides an appropriate objective for training a logistic regression classifier. MSE is generally better suited for regression problems where the target variable is continuous.

---
# 4. Neural Network From Scratch

A small multilayer perceptron (MLP) is implemented entirely from scratch using NumPy to demonstrate the core mechanics of a neural network without relying on machine learning frameworks.

The network contains two input features, one hidden layer with two neurons, and a single output neuron for binary classification. The implementation uses the sigmoid activation function and binary cross-entropy loss developed in the previous section.

Forward propagation, backpropagation, gradient computation, and parameter updates are implemented manually. A small 10-observation dataset is used so that the intermediate calculations and learned behavior can be examined clearly.

In [ ]:
# Small dataset used to demonstrate the neural network implementation
X_toy = np.array([
    [1, 5], [2, 6], [3, 4], [4, 7], [5, 8],
    [6, 7], [7, 8], [8, 3], [9, 6], [10, 4]
], dtype=float)

y_toy = np.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1], dtype=float)

student_names = ['A','B','C','D','E','F','G','H','I','J']
print(f"Toy dataset: {len(X_toy)} students")

## 4.1 Forward Propagation

In [ ]:
def forward(x, weights):
    """
    x: array (2,) — one data point [x1, x2]
    weights: dict with keys 'w11','w12','b1','w21','w22','b2','v1','v2','bo'
    Returns dict with 'z1','h1','z2','h2','zo','y_hat'
    """
    # Hidden neuron 1
    z1 = (weights['w11'] * x[0]) + (weights['w12'] * x[1]) + weights['b1']
    h1 = sigmoid(z1)
    # Hidden neuron 2
    z2 = (weights['w21'] * x[0]) + (weights['w22'] * x[1]) + weights['b2']
    h2 = sigmoid(z2)
    # Output neuron
    zo = (weights['v1'] * h1) + (weights['v2'] * h2) + weights['bo']
    y_hat = sigmoid(zo)
    return {'z1': z1, 'h1': h1, 'z2': z2, 'h2': h2, 'zo': zo, 'y_hat': y_hat}

In [ ]:
weights = {
    'w11': 1, 'w12': 0, 'b1': -3,
    'w21': 0, 'w22': 1, 'b2': -6,
    'v1': 1, 'v2': 1, 'bo': -1
}

print(f"{'Student':>8} {'x1':>4} {'x2':>4} | {'z1':>5} {'h1':>7} {'z2':>5} {'h2':>7} | {'y_hat':>7} {'Pred':>5} {'True':>5}")
print("-" * 75)
for i in range(len(X_toy)):
    r = forward(X_toy[i], weights)
    pred = 1 if r['y_hat'] >= 0.5 else 0
    mark = 'O' if pred == y_toy[i] else 'X'
    print(f"{student_names[i]:>8} {X_toy[i,0]:4.0f} {X_toy[i,1]:4.0f} | "
          f"{r['z1']:5.1f} {r['h1']:7.4f} {r['z2']:5.1f} {r['h2']:7.4f} | "
          f"{r['y_hat']:7.4f} {pred:5d} {int(y_toy[i]):5d} {mark}")

## 4.2 Backpropagation

Backpropagation is implemented manually to compute the gradients of the binary cross-entropy loss with respect to all nine parameters in the neural network.

The gradients are calculated by applying the chain rule from the output layer back through the hidden layer. These gradients are then used during training to update the network's weights and biases using gradient descent.

Implementing the backward pass from scratch demonstrates how prediction error is propagated through a neural network and how each parameter contributes to the final loss.

In [ ]:
def backward(x, y_true, fwd, weights):
    """Returns dict of gradients for all 9 parameters (one data point)."""
    # Step 1: output error
    dL_dzo = fwd['y_hat'] - y_true  

    # Step 2: output layer gradients
    dL_dv1 = dL_dzo * fwd['h1']
    dL_dv2 = dL_dzo * fwd['h2']
    dL_dbo = dL_dzo

    # Step 3: propagate to hidden layer
    dL_dh1 = dL_dzo * weights['v1'] 
    dL_dh2 = dL_dzo * weights['v2']

    # Step 4: through sigmoid
    dL_dz1 = dL_dh1 * sigmoid_derivative(fwd['z1'])
    dL_dz2 = dL_dh2 * sigmoid_derivative(fwd['z2'])

    # Step 5: hidden layer weight gradients
    dL_dw11 = dL_dz1 * x[0]
    dL_dw12 = dL_dz1 * x[1]
    dL_db1  = dL_dz1
    dL_dw21 = dL_dz2 * x[0]
    dL_dw22 = dL_dz2 * x[1]
    dL_db2  = dL_dz2

    return {
        'dv1': dL_dv1, 'dv2': dL_dv2, 'dbo': dL_dbo,
        'dw11': dL_dw11, 'dw12': dL_dw12, 'db1': dL_db1,
        'dw21': dL_dw21, 'dw22': dL_dw22, 'db2': dL_db2
    }

In [ ]:
fwd_F = forward(X_toy[5], weights)
grad_F = backward(X_toy[5], y_toy[5], fwd_F, weights)

print(f"y_hat = {fwd_F['y_hat']:.4f}, loss = {bce_loss(y_toy[5], fwd_F['y_hat']):.4f}")
print(f"Output error (dL/dzo): {fwd_F['y_hat'] - y_toy[5]:.4f}")
print(f"sigma'(z1) = {sigmoid_derivative(fwd_F['z1']):.4f}")
print(f"sigma'(z2) = {sigmoid_derivative(fwd_F['z2']):.4f}")
print("\nGradients:")
for name, val in grad_F.items():
    print(f"  {name:5s} = {val: .4f}")

## 4.3 Neural Network Training

The neural network is trained from randomly initialized weights using gradient descent. During each epoch, forward propagation generates predictions, backpropagation calculates the gradients, and the model parameters are updated to reduce binary cross-entropy loss.

Training loss is recorded throughout the process to visualize convergence. After training, the final predictions are compared with the actual outcomes for all 10 observations to evaluate the network's performance.

In [ ]:
def train_mlp(X, y, lr=0.5, epochs=1000):
    """Train the 2-2-1 MLP from random initialization. Returns (weights, loss_history)."""
    np.random.seed(42)
    w = {
        'w11': np.random.randn()*0.5, 'w12': np.random.randn()*0.5, 'b1': 0.0,
        'w21': np.random.randn()*0.5, 'w22': np.random.randn()*0.5, 'b2': 0.0,
        'v1':  np.random.randn()*0.5, 'v2':  np.random.randn()*0.5, 'bo': 0.0
    }
    loss_history = []
    for epoch in range(epochs):
        total_loss = 0
        grad_sum = {k: 0.0 for k in ['dv1','dv2','dbo','dw11','dw12','db1','dw21','dw22','db2']}
        for i in range(len(X)):
            fwd = forward(X[i], w)
            total_loss += bce_loss(y[i], fwd['y_hat'])
            grad = backward(X[i], y[i], fwd, w)
            for k in grad_sum:
                grad_sum[k] += grad[k]
        for k in grad_sum:
            grad_sum[k] /= len(X)

        w["v1"] -= lr * grad_sum["dv1"]
        w["v2"] -= lr * grad_sum["dv2"]
        w["bo"] -= lr * grad_sum["dbo"]

        w["w11"] -= lr * grad_sum["dw11"]
        w["w12"] -= lr * grad_sum["dw12"]
        w["b1"] -= lr * grad_sum["db1"]

        w["w21"] -= lr * grad_sum["dw21"]
        w["w22"] -= lr * grad_sum["dw22"]
        w["b2"] -= lr * grad_sum["db2"]

        avg_loss = total_loss / len(X)
        loss_history.append(avg_loss)
        if epoch % 100 == 0 or epoch == epochs - 1:
            correct = sum(1 for i in range(len(X))
                          if (forward(X[i], w)['y_hat'] >= 0.5) == y[i])
            print(f"Epoch {epoch:4d} | Loss: {avg_loss:.4f} | Accuracy: {correct}/{len(X)}")
    return w, loss_history

final_weights, losses = train_mlp(X_toy, y_toy, lr=0.5, epochs=1000)

print("\nFinal weights:")

for name, value in final_weights.items():
    print(f"{name}: {value:.4f}")

print("\nFinal predictions:")

print(
    f"{'Student':>8} {'Probability':>12} "
    f"{'Prediction':>12} {'True':>8}"
)

print("-" * 45)

correct = 0

for i in range(len(X_toy)):

    result = forward(
        X_toy[i],
        final_weights
    )

    probability = result["y_hat"]

    prediction = (
        1 if probability >= 0.5 else 0
    )

    if prediction == y_toy[i]:
        correct += 1

    print(
        f"{student_names[i]:>8} "
        f"{probability:>12.4f} "
        f"{prediction:>12} "
        f"{int(y_toy[i]):>8}"
    )

print(
    f"\nFinal accuracy: "
    f"{correct}/{len(X_toy)} "
    f"({correct / len(X_toy) * 100:.2f}%)"
)

plt.figure(figsize=(8, 5))

plt.plot(losses)

plt.xlabel("Epoch")
plt.ylabel("Average BCE Loss")
plt.title("Neural Network Training Loss")

plt.grid(True)
plt.show()

## 4.4 Neural Network Analysis

### Vanishing Gradients

The second hidden neuron received the smaller gradient because its sigmoid derivative was smaller. This demonstrates the vanishing gradient problem, where sigmoid neurons can produce very small gradients when their outputs become saturated, causing them to learn more slowly. An activation function such as ReLU can help reduce this problem because it maintains a non-zero gradient for positive inputs.

### Training Behavior

The loss generally decreased throughout training, with some fluctuations during the middle of the training process. The network eventually classified all 10 observations correctly, showing that the model successfully learned the small training dataset. The low final loss also indicates that the optimization process converged successfully.

---
# 5. Unsupervised Learning

Unsupervised learning techniques are used to identify patterns and natural groupings within the student data without using the target variables `exam_score` or `passed`.

The analysis uses the cleaned and normalized training features from the preprocessing stage. Multiple clustering approaches are explored, including K-Means, hierarchical clustering, and DBSCAN, to compare how different algorithms identify structure within the dataset.

After clustering is complete, the discovered groups are compared with the actual `passed` outcomes to evaluate whether the clusters capture patterns related to student performance. The target labels are used only for this final evaluation and are not involved in creating the clusters.

## 5.1 K-Means Clustering From Scratch

K-Means clustering is implemented from scratch using NumPy to identify natural groups within the normalized student performance features.

The algorithm initializes cluster centroids, assigns each observation to its nearest centroid using Euclidean distance, updates the centroids based on the assigned observations, and repeats the process until convergence. Within-cluster sum of squares (WCSS) is tracked to measure cluster compactness.

The custom implementation is then compared with scikit-learn's K-Means implementation to verify the clustering results.

In [ ]:
# === 5.1 K-Means from scratch ===
def kmeans_numpy(X, k, max_iters=100, random_state=42):
    """
    K-Means clustering using only NumPy.

    Returns:
        centroids: final cluster centers
        labels: cluster assignment for each row
        wcss_history: WCSS value from each iteration
    """

    rng = np.random.default_rng(random_state)

    # Randomly choose k existing rows as initial centroids
    initial_indices = rng.choice(
        len(X),
        size=k,
        replace=False
    )

    centroids = X[initial_indices].copy()

    wcss_history = []

    for _ in range(max_iters):

        # Step 1: Calculate Euclidean distance from
        # every row to every centroid
        distances = np.sqrt(
            ((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2)
            .sum(axis=2)
        )

        # Step 2: Assign each row to its nearest centroid
        labels = np.argmin(distances, axis=1)

        # Step 3: Calculate WCSS
        wcss = 0.0

        for cluster in range(k):
            cluster_points = X[labels == cluster]

            if len(cluster_points) > 0:
                wcss += np.sum(
                    (cluster_points - centroids[cluster]) ** 2
                )

        wcss_history.append(wcss)

        # Step 4: Recalculate centroids
        new_centroids = centroids.copy()

        for cluster in range(k):
            cluster_points = X[labels == cluster]

            if len(cluster_points) > 0:
                new_centroids[cluster] = cluster_points.mean(axis=0)

            else:
                # Empty-cluster handling:
                # replace it with a random data point
                random_index = rng.integers(len(X))
                new_centroids[cluster] = X[random_index]

        # Stop if centroids no longer change
        if np.allclose(centroids, new_centroids):
            centroids = new_centroids
            break

        centroids = new_centroids

    # Recalculate final labels using final centroids
    final_distances = np.sqrt(
        ((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2)
        .sum(axis=2)
    )

    labels = np.argmin(final_distances, axis=1)

    return centroids, labels, wcss_history

# Use normalized training features only
X_cluster = X_train_normalized.copy()

centroids, cluster_labels, wcss_history = kmeans_numpy(
    X_cluster,
    k=2,
    max_iters=100,
    random_state=42
)

print("Final normalized centroids:")

for cluster, centroid in enumerate(centroids):
    print(f"Cluster {cluster}: {centroid}")

print("\nCluster sizes:")

for cluster in range(2):
    cluster_size = np.sum(cluster_labels == cluster)
    print(f"Cluster {cluster}: {cluster_size} students")

#Save the cluster sizes for later comparison
kmeans_k2_cluster_sizes = [
    int(np.sum(cluster_labels == 0)),
    int(np.sum(cluster_labels == 1))
]

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(wcss_history) + 1),
    wcss_history,
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("WCSS")
plt.title("K-Means WCSS Across Iterations")
plt.grid(True)
plt.show()

sklearn_kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

sklearn_labels = sklearn_kmeans.fit_predict(X_cluster)

direct_differences = np.sum(
    cluster_labels != sklearn_labels
)

swapped_differences = np.sum(
    cluster_labels != (1 - sklearn_labels)
)

different_assignments = min(
    direct_differences,
    swapped_differences
)

print(
    "Number of differently assigned points:",
    different_assignments
)

original_centroids = (
    centroids * train_std
) + train_mean

centroid_table = pd.DataFrame(
    original_centroids,
    columns=feature_cols
)

centroid_table.index = [
    "Cluster 0",
    "Cluster 1"
]

display(centroid_table)

### Cluster Interpretation

**Cluster 0:** Represents students with lower overall academic performance indicators. These students studied fewer hours, had lower attendance rates, and had lower previous exam scores compared with students in Cluster 1.

**Cluster 1:** Represents students with stronger academic performance indicators. These students studied substantially more hours, had slightly higher attendance rates, and had much higher previous exam scores compared with students in Cluster 0.

Sleep hours were relatively similar between the two clusters, suggesting that this feature contributed less to distinguishing the two groups than study hours, attendance rate, and previous exam scores.


## 5.2 Elbow Method 

In [ ]:
# === 5.2 Elbow ===
k_values = range(1, 9)
wcss_values = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_cluster)

    # sklearn calls WCSS "inertia"
    wcss_values.append(model.inertia_)


# Display WCSS values
elbow_results = pd.DataFrame({
    "K": list(k_values),
    "WCSS": wcss_values
})

display(elbow_results)


# Plot the elbow curve
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    wcss_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method for Choosing K")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

chosen_k = 2

print("Chosen K:", chosen_k)

## 5.3 Hierarchical Clustering & DBSCAN 

In [ ]:
# === 5.3 Hierarchical + DBSCAN ===
linkage_matrix = linkage(
    X_cluster,
    method="ward"
)

plt.figure(figsize=(12, 6))

dendrogram(
    linkage_matrix,
    no_labels=True
)

plt.xlabel("Students")
plt.ylabel("Ward Distance")
plt.title("Hierarchical Clustering Dendrogram")
plt.grid(axis="y")
plt.show()

agglomerative_model = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward"
)

agglomerative_labels = agglomerative_model.fit_predict(
    X_cluster
)

print(
    "Number of agglomerative clusters:",
    len(np.unique(agglomerative_labels))
)

print(
    "Agglomerative cluster sizes:",
    np.bincount(agglomerative_labels)
)


eps_values = [0.5, 0.8, 1.2]

dbscan_results = []

for eps in eps_values:

    dbscan = DBSCAN(
        eps=eps,
        min_samples=5
    )

    dbscan_labels = dbscan.fit_predict(
        X_cluster
    )

    if eps == 0.8:
        best_dbscan_labels = dbscan_labels.copy()



    # Number of clusters (ignore -1 because it's noise)
    unique_clusters = set(dbscan_labels)
    unique_clusters.discard(-1)

    num_clusters = len(unique_clusters)

    # Number of noise points
    noise_points = np.sum(dbscan_labels == -1)

    # Largest cluster size
    if num_clusters > 0:
        largest_cluster = max(
            np.sum(dbscan_labels == c)
            for c in unique_clusters
        )
    else:
        largest_cluster = 0

    dbscan_results.append({
        "eps": eps,
        "Clusters": num_clusters,
        "Noise Points": noise_points,
        "Largest Cluster": largest_cluster
    })

dbscan_table = pd.DataFrame(dbscan_results)

display(dbscan_table)

# Select the best eps (middle-ground choice)
best_dbscan_row = dbscan_table.loc[
    dbscan_table["eps"] == 0.8
]

dbscan_best_eps = float(best_dbscan_row["eps"].iloc[0])

print("Chosen DBSCAN eps:", dbscan_best_eps)

noise_indices = np.where(best_dbscan_labels == -1)[0]

print("Noise point positions in the training data:")
print(noise_indices[:10])

noise_students = pd.DataFrame(
    X_train[noise_indices[:3]],
    columns=feature_cols
)

display(noise_students)


### Clustering Method Comparison

The hierarchical clustering analysis produced two main clusters, which agrees with the K = 2 selected using the elbow method for K-Means. Both methods therefore suggest that the student data contains two major groups based on the normalized features.

DBSCAN produced different results depending on the value of `eps`. At `eps = 0.5`, the algorithm identified 4 clusters but classified 217 observations as noise, suggesting that the neighborhood radius was too restrictive. Increasing `eps` to 0.8 resulted in 1 cluster with 52 noise points, while `eps = 1.2` placed all 240 observations into a single cluster with no noise points.

An `eps` value of 0.8 was selected because it provided a balance between identifying a dense group of students and detecting observations that differed from the main pattern. The noise points represent students with combinations of features that are less common compared with the majority of the dataset.

These results demonstrate an important difference between the clustering methods. K-Means and hierarchical clustering divide the observations into predefined or selected groups, while DBSCAN can identify dense regions and classify observations outside those regions as noise.

## 5.4 Cluster vs. Ground Truth

In [ ]:
# === 5.4 Ground truth comparison ===
# True labels for the training rows
y_true = clean_df.iloc[train_indices]["passed"].to_numpy()

# Crosstab
comparison = pd.crosstab(
    pd.Series(cluster_labels, name="Cluster"),
    pd.Series(y_true, name="Passed")
)

display(comparison)

# Direct mapping
alignment1 = np.mean(cluster_labels == y_true)

# Swapped mapping
alignment2 = np.mean((1 - cluster_labels) == y_true)

alignment = max(alignment1, alignment2)

print(f"Cluster alignment: {alignment*100:.2f}%")

plt.figure(figsize=(12,5))

# Cluster labels
plt.subplot(1,2,1)

plt.scatter(
    X_train_normalized[:,0],
    X_train_normalized[:,1],
    c=cluster_labels,
    cmap="viridis"
)

plt.xlabel(feature_cols[0])
plt.ylabel(feature_cols[1])
plt.title("Colored by K-Means Cluster")

# True labels
plt.subplot(1,2,2)

plt.scatter(
    X_train_normalized[:,0],
    X_train_normalized[:,1],
    c=y_true,
    cmap="viridis"
)

plt.xlabel(feature_cols[0])
plt.ylabel(feature_cols[1])
plt.title("Colored by True Passed Label")

plt.tight_layout()
plt.show()

### Cluster Evaluation

K-Means was able to identify groups that aligned with the actual pass/fail outcomes at approximately 82.08%, despite not using the `passed` labels during clustering. This suggests that the selected student performance features contain meaningful structure related to academic outcomes.

The alignment was not perfect because students with similar feature values do not always have the same outcome. For example, a student may study many hours but still fail, while another student may study fewer hours and pass. This overlap makes it difficult for an unsupervised algorithm to perfectly separate the two groups.

Unsupervised learning can be especially useful when labels are unavailable, incomplete, or expensive to obtain because it can still identify patterns and natural groupings within the data.

# 6. Results

The project produced consistent results between the custom NumPy implementations and their library-based counterparts.

| Model / Analysis | Result |
|---|---|
| Linear Regression (NumPy) | Validation MSE: 38.18 |
| Linear Regression (PyTorch) | Validation MSE: 38.18 |
| Best Polynomial Regression | Degree 2 |
| Logistic Regression (NumPy) | Validation Accuracy: 90.16% |
| Logistic Regression (PyTorch) | Validation Accuracy: 90.16% |
| Neural Network | 10/10 training examples classified correctly |
| K-Means | 2 clusters |
| Elbow Method | K = 2 |
| DBSCAN | Selected ε = 0.8 |
| K-Means vs. Pass/Fail | 82.08% alignment |

The close agreement between the NumPy and PyTorch implementations helps validate the from-scratch regression models. The clustering results also revealed meaningful structure in the student data, with K-Means producing two clusters that aligned with actual pass/fail outcomes at approximately 82.08%.

# 7. Conclusion

This project demonstrated a complete machine learning workflow, beginning with data cleaning and preprocessing and progressing through supervised and unsupervised learning techniques. Feature selection and normalization were especially important because they affected both gradient-based models and distance-based clustering algorithms.

Implementing linear regression, logistic regression, neural network backpropagation, and K-Means from scratch with NumPy provided a deeper understanding of how these algorithms operate internally. Comparing the custom implementations with PyTorch and scikit-learn also helped verify that the implementations were producing consistent results.

The supervised models showed strong predictive performance, while the clustering analysis revealed meaningful structure in the student data without using the target labels. K-Means produced clusters that aligned with actual pass/fail outcomes at approximately 82.08%, demonstrating that the selected features contained patterns related to student performance.

One of the most valuable parts of the project was seeing how preprocessing, model complexity, optimization, and feature scaling influence the final results. Future improvements could include testing additional models, performing more extensive hyperparameter tuning, and evaluating the models on larger datasets.